# YOLO Parking Spot Detection - Google Colab Training

This notebook trains a YOLOv8 model for parking spot detection on Google Colab with GPU support.

## Setup Instructions:
1. **Enable GPU**: Go to `Runtime → Change runtime type → Hardware accelerator → GPU (T4)`
2. **Upload Dataset**: Choose one of the methods below
3. **Run all cells**: `Runtime → Run all`

---

## 1. Check GPU Availability

In [5]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA device count: {torch.cuda.device_count()}")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → GPU")

PyTorch version: 2.8.0+cu126
CUDA available: False
⚠️ No GPU detected! Go to Runtime → Change runtime type → GPU


In [6]:
try:
    device = xm.xla_device()
    print(f"TPU detected: {device}")
    # To get more specific details about the TPU cores
    import torch_xla.runtime as xr
    print(f"TPU Runtime Type: {xr.device_type()}")
except Exception as e:
    print("⚠️ TPU not detected. Ensure your runtime is set to TPU.")

⚠️ TPU not detected. Ensure your runtime is set to TPU.


## 2. Install Dependencies

In [ ]:
!pip install -q ultralytics mlflow
print("✅ Dependencies installed!")

## 3. Upload Dataset

### Option A: Upload from Google Drive (Recommended)
Upload your dataset folder to Google Drive first, then mount it.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# # Set the path to your dataset in Google Drive
# # Example: DATASET_PATH = '/content/drive/MyDrive/parking_spot_dataset'
# DATASET_PATH = '/content/drive/MyDrive/parking_spot_dataset'  # ⚠️ CHANGE THIS PATH

# print(f"Dataset path set to: {DATASET_PATH}")

### Option B: Upload ZIP file directly (Alternative)

In [ ]:
# Uncomment and run this if you want to upload a ZIP file
# from google.colab import files
# import zipfile
# import os

# print("Upload your dataset ZIP file...")
# uploaded = files.upload()

# for filename in uploaded.keys():
#     print(f"Extracting {filename}...")
#     with zipfile.ZipFile(filename, 'r') as zip_ref:
#         zip_ref.extractall('/content/')

# DATASET_PATH = '/content/parking_spot_dataset'  # Adjust to your extracted folder name
# print(f"Dataset extracted to: {DATASET_PATH}")

In [3]:
# !git clone https://github.com/ReggieReo/ku-parking-ai-component.git
# %cd ku-parking-ai-component
# !git checkout parking_space
# !ls -a

Cloning into 'ku-parking-ai-component'...
fatal: unable to access 'https://github.com/ReggieReo/ku-parking-ai-component.git/': Could not resolve host: github.com
[Errno 2] No such file or directory: 'ku-parking-ai-component'
/kaggle/working
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
.  ..  .virtual_documents


In [4]:
DATASET_PATH = '/kaggle/input/test-parking-spot/parking_spot_dataset'

## 4. Verify Dataset Structure

In [5]:
import os

# Check if dataset exists
if not os.path.exists(DATASET_PATH):
    print(f"❌ Dataset not found at {DATASET_PATH}")
    print("Please check your path and try again!")
else:
    print(f"✅ Dataset found at {DATASET_PATH}")
    print("\nDataset structure:")
    for root, dirs, files in os.walk(DATASET_PATH):
        level = root.replace(DATASET_PATH, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 2 * (level + 1)
        for file in files[:5]:  # Show first 5 files
            print(f"{subindent}{file}")
        if len(files) > 5:
            print(f"{subindent}... and {len(files) - 5} more files")
        if level > 2:  # Limit depth
            break

✅ Dataset found at /content/ku-parking-ai-component/parking_spot_dataset

Dataset structure:
parking_spot_dataset/
  data.yaml
  .DS_Store
  labels/
    val.cache
    .DS_Store
    train.cache
    val/
      180227-143842CAM1_000573.txt
      180814-143248CAM3_000345.txt
      180523-154911CAM3_000104.txt
      180523-145908CAM3_000884.txt
      180711-103610CAM2_001668.txt
      ... and 4513 more files
    train/
      180827-201031CAM2_001542.txt
      180308-132457CAM1_000233.txt
      180227-102425CAM1_001193.txt
      180313-143821CAM1_000227.txt
      180523-155339CAM2_000072.txt
      ... and 18294 more files
  images/
    .DS_Store
    val/
      180503-124022CAM3_000209.jpg
      180227-141952CAM1_000394.jpg
      180503-103347CAM2_000261.jpg
      180123_02_000975.jpg
      190531_000012CAM2_000628.jpg
      ... and 4514 more files
    train/
      180827-164651CAM2_003246.jpg
      180227-140033CAM1_002574.jpg
      180711-104253CAM3_001714.jpg
      180711-144336CAM3_001950

## 5. Create/Verify data.yaml Configuration

In [6]:
import yaml

# Create data.yaml if it doesn't exist
data_yaml_path = os.path.join(DATASET_PATH, 'data.yaml')

if not os.path.exists(data_yaml_path):
    print("Creating data.yaml...")
    data_yaml_content = f"""path: {DATASET_PATH}
train: images/train
val: images/val

# Number of classes
nc: 1

# Class names
names: ["parking_space"]
"""
    with open(data_yaml_path, 'w') as f:
        f.write(data_yaml_content.strip())
    print(f"✅ data.yaml created at {data_yaml_path}")
else:
    print("✅ data.yaml already exists")

# Display data.yaml content
print("\ndata.yaml content:")
with open(data_yaml_path, 'r') as f:
    print(f.read())

DATA_YAML = data_yaml_path

✅ data.yaml already exists

data.yaml content:
# dataset/data.yaml
train: images/train
val: images/val

# Number of classes
nc: 1

# Class names
names: ["parking_space"]


## 6. Configure Training Parameters

In [16]:
import itertools

# --- Configuration ---
PRETRAINED_MODEL_NAME = "yolo26n.pt"  # Options: yolo11n.pt, yolo11s.pt, yolo11m.pt, etc.
OUTPUT_DIR = "yolo_parking_outputs"
IMAGE_SIZE = 640

# --- Hyperparameter Grid ---
param_grid = {
    "optimizer": ["auto"],
    "batch": [64],  # Adjust based on GPU memory (16 for better GPU, 4 for limited memory)
    "weight_decay": [0.0005],
    "epochs": [150],
    "patience": [50],
    "lr0": [0.01],
}

keys, values = zip(*param_grid.items())
hyperparameter_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

print(f"Total training combinations: {len(hyperparameter_combinations)}")
print("\nCombinations:")
for i, params in enumerate(hyperparameter_combinations, 1):
    print(f"{i}. {params}")

Total training combinations: 1

Combinations:
1. {'optimizer': 'auto', 'batch': 64, 'weight_decay': 0.0005, 'epochs': 150, 'patience': 50, 'lr0': 0.01}


## 7. Setup MLflow Tracking

In [17]:
import mlflow
from ultralytics import settings

# Enable MLflow integration in Ultralytics
settings.update({"mlflow": True})

# Setup MLflow to use local file storage (Colab-friendly)
MLFLOW_DIR = "/content/mlruns"
mlflow.set_tracking_uri(f"file://{MLFLOW_DIR}")

EXPERIMENT_NAME = "yolo_parking_spot_colab"
experiment = mlflow.set_experiment(EXPERIMENT_NAME)

print(f"✅ MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"✅ MLflow Experiment: {experiment.name}")
print(f"✅ MLflow Experiment ID: {experiment.experiment_id}")
print(f"\nℹ️  Artifacts will be saved to: {experiment.artifact_location}")

✅ MLflow Tracking URI: file:///content/mlruns
✅ MLflow Experiment: yolo_parking_spot_colab
✅ MLflow Experiment ID: 444831193467210356

ℹ️  Artifacts will be saved to: file:///content/mlruns/444831193467210356


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

## 8. Run Training (Grid Search)

⚠️ **This cell will take time to complete!** Training 4 models (epochs: 50, 100 for each weight_decay value)

In [ ]:
from ultralytics import YOLO
import os

# Device selection (should be cuda on Colab with GPU)
if torch.cuda.is_available():
    device_to_use = "cuda"
elif torch.backends.mps.is_available():
    device_to_use = "mps"
else:
    device_to_use = "cpu"

print(f"🚀 Using device: {device_to_use}")
print(f"\n🏋️ Starting training with {len(hyperparameter_combinations)} combinations...\n")
print("=" * 80)

# Training loop
for i, params in enumerate(hyperparameter_combinations):
    run_name_parts = [f"run_{i+1}"]
    for key, value in sorted(params.items()):
        run_name_parts.append(f"{key}-{value}")
    run_name = "_".join(run_name_parts)

    # End any active MLflow run
    active_mlflow_run = mlflow.active_run()
    if active_mlflow_run is not None:
        mlflow.end_run()

    print(f"\n{'='*80}")
    print(f"🔄 Run {i+1}/{len(hyperparameter_combinations)}: {run_name}")
    print(f"{'='*80}")
    print(f"Parameters: {params}")

    with mlflow.start_run(run_name=run_name) as run:
        # Log parameters
        mlflow.log_param("pretrained_model", PRETRAINED_MODEL_NAME)
        mlflow.log_param("image_size", IMAGE_SIZE)
        mlflow.log_param("data_yaml", DATA_YAML)
        mlflow.log_param("device_used", device_to_use)

        for param_name, param_value in params.items():
            mlflow.log_param(param_name, param_value)

        try:
            # Initialize and train model
            model = YOLO(PRETRAINED_MODEL_NAME)
            results = model.train(
                data=DATA_YAML,
                imgsz=IMAGE_SIZE,
                project=OUTPUT_DIR,
                name=run_name,
                device=device_to_use,
                exist_ok=True,
                **params
            )

            # Log artifacts
            run_output_dir = os.path.join(OUTPUT_DIR, run_name)
            best_model_path = os.path.join(run_output_dir, "weights", "best.pt")

            if os.path.exists(best_model_path):
                mlflow.log_artifact(best_model_path, artifact_path="yolo_model_weights")
                print(f"✅ Best model logged: {best_model_path}")
            else:
                print(f"⚠️ Best model not found at {best_model_path}")

            # Log training artifacts
            artifacts_to_log = [
                "results.csv", "results.png", "confusion_matrix.png",
                "F1_curve.png", "PR_curve.png", "P_curve.png",
                "R_curve.png", "labels.jpg", "labels_correlogram.jpg",
                "val_batch0_labels.jpg", "val_batch0_pred.jpg",
            ]

            for item_name in artifacts_to_log:
                item_path = os.path.join(run_output_dir, item_name)
                if os.path.exists(item_path):
                    mlflow.log_artifact(item_path, artifact_path="training_outputs")

            mlflow.set_tag("run_status", "completed")
            print(f"\n✅ Run {run_name} completed successfully!")

        except Exception as e:
            print(f"\n❌ Error during run {run_name}: {e}")
            try:
                mlflow.set_tag("run_status", "failed")
                mlflow.log_param("error_message", str(e))
            except Exception as log_err:
                print(f"Error logging failure to MLflow: {log_err}")
            continue

print(f"\n{'='*80}")
print("🎉 Grid search training completed!")
print(f"{'='*80}")

🚀 Using device: cuda

🏋️ Starting training with 1 combinations...


🔄 Run 1/1: run_1_batch-64_epochs-150_lr0-0.01_optimizer-auto_patience-50_weight_decay-0.0005
Parameters: {'optimizer': 'auto', 'batch': 64, 'weight_decay': 0.0005, 'epochs': 150, 'patience': 50, 'lr0': 0.01}
Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ku-parking-ai-component/parking_spot_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, k

2026/01/27 08:00:10 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/01/27 08:00:10 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.


MLflow: logging run_id(4713b323e0914c889d108da4ddb0b2e0) to file:///content/mlruns
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /content/ku-parking-ai-component/runs/detect/yolo_parking_outputs/run_1_batch-64_epochs-150_lr0-0.01_optimizer-auto_patience-50_weight_decay-0.0005
Starting training for 150 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


## 9. View Training Results

In [ ]:
import pandas as pd
from IPython.display import display

# List all runs
print("📊 Training Runs Summary:\n")
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

if len(runs) > 0:
    # Display key metrics
    columns_to_show = ['run_id', 'start_time', 'tags.run_status',
                       'params.epochs', 'params.batch', 'params.weight_decay']
    # Add metric columns if they exist
    metric_cols = [col for col in runs.columns if col.startswith('metrics.')]
    columns_to_show.extend(metric_cols[:5])  # Show first 5 metrics

    display_cols = [col for col in columns_to_show if col in runs.columns]
    print(runs[display_cols].to_string())
else:
    print("No runs found.")

# Show training outputs
print("\n📁 Training Output Directories:")
if os.path.exists(OUTPUT_DIR):
    for run_dir in os.listdir(OUTPUT_DIR):
        print(f"  - {OUTPUT_DIR}/{run_dir}")
else:
    print(f"  No output directory found at {OUTPUT_DIR}")

## 10. Display Training Curves (Latest Run)

In [ ]:
from IPython.display import Image, display
import os

# Get the latest run directory
if os.path.exists(OUTPUT_DIR):
    run_dirs = sorted([d for d in os.listdir(OUTPUT_DIR) if os.path.isdir(os.path.join(OUTPUT_DIR, d))])
    if run_dirs:
        latest_run = run_dirs[-1]
        latest_run_path = os.path.join(OUTPUT_DIR, latest_run)

        print(f"📈 Displaying results from: {latest_run}\n")

        # Display key images
        images_to_display = [
            ("results.png", "Training Results"),
            ("confusion_matrix.png", "Confusion Matrix"),
            ("F1_curve.png", "F1 Score Curve"),
            ("PR_curve.png", "Precision-Recall Curve"),
            ("val_batch0_pred.jpg", "Validation Predictions"),
        ]

        for img_name, title in images_to_display:
            img_path = os.path.join(latest_run_path, img_name)
            if os.path.exists(img_path):
                print(f"\n{'='*60}")
                print(f"📊 {title}")
                print(f"{'='*60}")
                display(Image(filename=img_path, width=800))
            else:
                print(f"⚠️ {img_name} not found")
    else:
        print("No training runs found.")
else:
    print(f"Output directory {OUTPUT_DIR} not found.")

## 11. Test the Best Model

In [ ]:
# Load the best model from the latest run
if os.path.exists(OUTPUT_DIR):
    run_dirs = sorted([d for d in os.listdir(OUTPUT_DIR) if os.path.isdir(os.path.join(OUTPUT_DIR, d))])
    if run_dirs:
        latest_run = run_dirs[-1]
        best_model_path = os.path.join(OUTPUT_DIR, latest_run, "weights", "best.pt")

        if os.path.exists(best_model_path):
            print(f"✅ Loading best model from: {best_model_path}\n")
            best_model = YOLO(best_model_path)

            # Test on a validation image
            val_images_dir = os.path.join(DATASET_PATH, "images", "val")
            if os.path.exists(val_images_dir):
                val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
                if val_images:
                    test_image = os.path.join(val_images_dir, val_images[0])
                    print(f"🔍 Testing on: {val_images[0]}\n")

                    results = best_model.predict(test_image, conf=0.25, save=True)

                    # Display prediction
                    if results and len(results) > 0:
                        print(f"\n✅ Detection complete!")
                        print(f"   Detected objects: {len(results[0].boxes)}")

                        # The prediction image is saved in runs/detect/predict
                        pred_img_path = "runs/detect/predict/" + val_images[0]
                        if os.path.exists(pred_img_path):
                            print(f"\n📸 Prediction Result:")
                            display(Image(filename=pred_img_path, width=800))
                else:
                    print("No validation images found.")
            else:
                print(f"Validation directory not found: {val_images_dir}")
        else:
            print(f"Best model not found at {best_model_path}")
    else:
        print("No training runs found.")
else:
    print(f"Output directory {OUTPUT_DIR} not found.")

## 12. Download Trained Models to Local Machine

In [ ]:
# Option A: Download best model directly
from google.colab import files

if os.path.exists(OUTPUT_DIR):
    run_dirs = sorted([d for d in os.listdir(OUTPUT_DIR) if os.path.isdir(os.path.join(OUTPUT_DIR, d))])
    if run_dirs:
        print("📦 Available models to download:\n")
        for i, run_dir in enumerate(run_dirs, 1):
            best_model_path = os.path.join(OUTPUT_DIR, run_dir, "weights", "best.pt")
            if os.path.exists(best_model_path):
                print(f"{i}. {run_dir}")
                # Uncomment the next line to download
                # files.download(best_model_path)

        print("\n💡 Uncomment files.download() lines in the code to download models.")
        print("💡 Or copy the output to Google Drive for persistent storage.")
    else:
        print("No trained models found.")
else:
    print(f"Output directory {OUTPUT_DIR} not found.")

## 13. (Optional) Copy Results to Google Drive

In [ ]:
import shutil

# Copy entire output directory to Google Drive
drive_output_path = '/content/drive/MyDrive/yolo_training_results'

if os.path.exists(OUTPUT_DIR):
    print(f"📁 Copying {OUTPUT_DIR} to Google Drive...")
    try:
        if os.path.exists(drive_output_path):
            shutil.rmtree(drive_output_path)
        shutil.copytree(OUTPUT_DIR, drive_output_path)
        print(f"✅ Results saved to: {drive_output_path}")
    except Exception as e:
        print(f"❌ Error copying to Drive: {e}")
else:
    print(f"Output directory {OUTPUT_DIR} not found.")

# Also copy MLflow artifacts
if os.path.exists(MLFLOW_DIR):
    drive_mlflow_path = '/content/drive/MyDrive/yolo_mlflow_results'
    print(f"\n📁 Copying MLflow results to Google Drive...")
    try:
        if os.path.exists(drive_mlflow_path):
            shutil.rmtree(drive_mlflow_path)
        shutil.copytree(MLFLOW_DIR, drive_mlflow_path)
        print(f"✅ MLflow results saved to: {drive_mlflow_path}")
    except Exception as e:
        print(f"❌ Error copying MLflow to Drive: {e}")

---

## 🎉 Training Complete!

### Next Steps:
1. **Review Results**: Check the training curves and metrics above
2. **Download Models**: Use cell 12 to download trained models
3. **Save to Drive**: Use cell 13 to backup results to Google Drive
4. **Test Locally**: Use the downloaded `.pt` file with your local inference script

### Useful Commands for Local Testing:
```python
from ultralytics import YOLO

# Load your trained model
model = YOLO('path/to/best.pt')

# Run inference
results = model.predict('image.jpg', conf=0.25)
results[0].show()
```

---